# 과거 유사 충격 비교 자료 내보내기
기본 Lakehouse: LH_StressLab_ML. NB_05와 같은 Environment를 사용합니다.
기존 모델의 학습 Gold 해시와 Train 표본 수·기간을 검증합니다.
학습·보정·테스트를 다시 수행하지 않고, 기존 Gold·모델·스냅샷·PostgreSQL을 변경하지 않습니다.
모델 패키지의 기록된 종료일을 사용하므로 날짜를 현재 날짜로 바꾸지 마세요.
마지막 셀은 새 Files/stresslab/conditional_v4/history 경로에 비교 자료만 저장합니다.


In [ ]:
GOLD_TABLE = "lh_whatif.gold_stress_daily"
PACKAGE_PATH = "/lakehouse/default/Files/stresslab/conditional_v4/packages/hab-conditional-v4-c217c900393e7e49.zip"
EXPECTED_ZIP_SHA256 = "c7a1d0c53cb6680a8dffee78130d6d4e4c7dda835f6f2478d90425a58957808d"
EXPECTED_MANIFEST_SHA256 = "56fcb33f620b475fb84994ecd940574b38bfc80ec7a8ec7899f4d4c82e478fda"


In [ ]:
import sys,types,json,hashlib
from pathlib import Path
SOURCE='from pathlib import Path\nimport base64,hashlib,importlib,importlib.metadata,io,json,os,sys,tempfile,zipfile,types\nfrom datetime import datetime,timezone\nimport numpy as np,pandas as pd,joblib\nSOURCE_SHA256 = "c545beda9904701fdc97bdb3b96f219a6cafc7650373f853b2cf9c23aa212dc8"\nSOURCE_B64 = "..."\nsource_bytes = base64.b64decode(SOURCE_B64)\nif hashlib.sha256(source_bytes).hexdigest() != SOURCE_SHA256:\n    raise ValueError("내장 소스 SHA256 불일치")\nWORKDIR = Path(tempfile.mkdtemp(prefix="stresslab_compare_"))\nwith zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:\n    for name in archive.namelist():\n        if not (WORKDIR / name).resolve().is_relative_to(WORKDIR.resolve()):\n            raise ValueError("잘못된 소스 경로")\n    archive.extractall(WORKDIR)\nfor part in ("src","model_de","model_huber_interactions","model_catboost_uncertainty"):\n    sys.path.insert(0, str(WORKDIR / "analytics" / part))\nfor name in ("common","conformal","distribution","features","targets","splits","horizon_samples",\n             "load_gold","hybrid","huber_core","catboost_core","prep","models","barc","hab_core","hab_lightgbm_reference","hab_catboost_core","huber_catboost_hybrid_core"):\n    sys.modules.pop(name, None)\nimport common as C\nfrom load_gold import load_gold\nSCHEMA = WORKDIR / "contracts/daily/schema.json"\nprint("원본 커밋", "eede149fe8ac09be9e4529effbf391ae6b55f7af", "소스 SHA256", SOURCE_SHA256)\n\nsources_archive = zipfile.ZipFile(io.BytesIO(source_bytes))\nI=types.ModuleType(\'conditional_inference\')\nexec(compile(\'"""Conditional Huber+CatBoost inference, matching the selected research model.\\n\\nThe historical evaluation used realized future market moves. Service requests\\nmust supply assumed moves, separately for each 1/5/20-trading-day horizon.\\n"""\\nfrom pathlib import Path\\nimport hashlib\\nimport json\\nimport math\\nimport importlib.metadata\\nimport numpy as np\\nimport pandas as pd\\nimport joblib\\n\\nHORIZONS=(1,5,20)\\nSHOCKS=[\\\'shock_usdkrw_pct\\\',\\\'shock_us10y_bp\\\',\\\'shock_wti_pct\\\',\\\'shock_vix_pt\\\']\\nSTATES=[\\\'kospi_vol20\\\',\\\'kospi_trend20\\\',\\\'vix_level\\\',\\\'us10y_level\\\',\\\'usdkrw_trend20\\\',\\\'wti_level\\\']\\nSTATE_FIELDS=STATES+[\\\'scale_kospi\\\',\\\'asset_scale\\\',\\\'beta_shr\\\',\\\'beta_dn_f\\\',\\\'beta_up_f\\\',\\n                    \\\'idio_f\\\',\\\'asset_vol20\\\',\\\'asset_vol60\\\']\\nMODEL=\\\'huber_catboost_hybrid_conditional\\\'\\n\\ndef design(frame):\\n    ks=np.maximum(frame.scale_kospi.to_numpy(float),.05)\\n    scale=np.maximum(frame.asset_scale.to_numpy(float),.05)\\n    base=np.column_stack([frame[c].to_numpy(float)/ks if c in SHOCKS else frame[c].to_numpy(float)\\n                          for c in SHOCKS+STATES])\\n    # Preserve the exact order of barc.feat() and the selected hybrid design().\\n    extra=np.column_stack([frame.m/ks,frame.m/scale,np.abs(frame.m)/ks,frame.beta_shr,\\n                           frame.beta_dn_f-frame.beta_up_f,frame.idio_f/scale*np.sqrt(frame.horizon),\\n                           frame.asset_vol20/frame.asset_vol60.replace(0,np.nan).fillna(1.)])\\n    interactions=(base[:,:4,None]*base[:,None,4:]).reshape(len(frame),-1)\\n    result=np.column_stack([base,extra,interactions])\\n    if result.shape[1]!=41 or not np.isfinite(result).all():\\n        raise ValueError(\\\'Conditional model requires 41 finite design features\\\')\\n    return result\\n\\ndef predict(model,frame):\\n    x=design(frame);scale=np.maximum(frame.asset_scale.to_numpy(float),.05)\\n    huber=model[\\\'huber\\\'].predict(x)*scale\\n    residual=model[\\\'catboost\\\'].predict(x,prediction_type=\\\'RawFormulaVal\\\')\\n    center=huber+residual[:,0]*scale;sigma=np.exp(residual[:,1])*scale\\n    result=pd.DataFrame(dict(center=center,center_huber=huber,\\n                            catboost_correction=residual[:,0]*scale,sigma=sigma),index=frame.index)\\n    for level,(below,above) in model[\\\'bands\\\'].items():\\n        tag=int(round(level*100));result[f\\\'lo_{tag}\\\']=center-below*sigma;result[f\\\'hi_{tag}\\\']=center+above*sigma\\n    if not np.isfinite(result.to_numpy()).all() or (sigma<=0).any():\\n        raise ValueError(\\\'Invalid conditional output\\\')\\n    return result\\n\\ndef scenario_frame(state,scenario):\\n    expected={\\\'horizon_days\\\',\\\'kospi_pct\\\',*SHOCKS}\\n    if set(scenario)!=expected:\\n        raise ValueError(\\\'Each horizon requires KOSPI and all four macro assumptions\\\')\\n    for k,v in scenario.items():\\n        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v):\\n            raise ValueError(\\\'Scenario must contain finite numbers\\\')\\n    if scenario[\\\'horizon_days\\\'] not in HORIZONS:\\n        raise ValueError(\\\'Unsupported horizon\\\')\\n    for k in (\\\'kospi_pct\\\',\\\'shock_usdkrw_pct\\\',\\\'shock_wti_pct\\\'):\\n        if scenario[k]<=-100:raise ValueError(k+\\\' must be greater than -100 percent\\\')\\n    result=pd.DataFrame(state).copy()\\n    if not result.horizon.eq(scenario[\\\'horizon_days\\\']).all():raise ValueError(\\\'State horizon mismatch\\\')\\n    for c in SHOCKS:result[c]=float(scenario[c])\\n    result[\\\'m\\\']=100*np.log1p(scenario[\\\'kospi_pct\\\']/100)\\n    return result\\n\\ndef scenario_outputs(models,snapshot,scenarios,asset_id=None):\\n    if len(scenarios)!=3 or {s[\\\'horizon_days\\\'] for s in scenarios}!={1,5,20}:\\n        raise ValueError(\\\'Supply one scenario for each of 1,5,20 trading days\\\')\\n    records=[]\\n    for scenario in sorted(scenarios,key=lambda s:s[\\\'horizon_days\\\']):\\n        h=scenario[\\\'horizon_days\\\'];state=snapshot[\\\'state_by_horizon\\\'][str(h)]\\n        if asset_id is not None:state=[r for r in state if r[\\\'asset_id\\\']==asset_id]\\n        if not state:raise ValueError(\\\'Unsupported asset\\\')\\n        frame=scenario_frame(state,scenario)\\n        z=predict(models[h],frame)\\n        for i,row in frame.iterrows():\\n            p=z.loc[i];close=float(row.base_close)\\n            record=dict(asset_id=row.asset_id,horizon_days=h,base_close=close,\\n                        point_log_return_pct=float(p.center),point_simple_return_pct=float(100*np.expm1(p.center/100)),\\n                        predicted_close=float(close*np.exp(p.center/100)))\\n            for tag in (80,90):\\n                for side,short in [(\\\'lower\\\',\\\'lo\\\'),(\\\'upper\\\',\\\'hi\\\')]:\\n                    value=float(p[f\\\'{short}_{tag}\\\'])\\n                    record[f\\\'{side}_{tag}_log_return_pct\\\']=value\\n                    record[f\\\'{side}_{tag}_close\\\']=float(close*np.exp(value/100))\\n            if not all(isinstance(v,str) or np.isfinite(v) for v in record.values()):raise ValueError(\\\'Nonfinite price output\\\')\\n            records.append(record)\\n    return records\\n\\ndef load_models(root,expected_manifest_sha256):\\n    root=Path(root);content=(root/\\\'manifest.json\\\').read_bytes()\\n    if hashlib.sha256(content).hexdigest()!=expected_manifest_sha256:raise ValueError(\\\'Manifest SHA mismatch\\\')\\n    manifest=json.loads(content)\\n    if (manifest.get(\\\'model\\\')!=MODEL or manifest.get(\\\'future_market_or_macro_input\\\') is not True\\n        or manifest.get(\\\'research_only\\\') is not True or manifest.get(\\\'production_approval\\\')!=\\\'NOT_APPROVED\\\'):\\n        raise ValueError(\\\'Conditional package contract mismatch\\\')\\n    for name in (\\\'numpy\\\',\\\'pandas\\\',\\\'scikit-learn\\\',\\\'catboost\\\',\\\'joblib\\\'):\\n        if importlib.metadata.version(name)!=manifest[\\\'versions\\\'][name]:\\n            raise ValueError(\\\'Model/API library versions differ: \\\'+name)\\n    models={}\\n    for h in HORIZONS:\\n        name=f\\\'model_{h}d.joblib\\\'\\n        if hashlib.sha256((root/name).read_bytes()).hexdigest()!=manifest[\\\'files\\\'][name]:raise ValueError(\\\'Model SHA mismatch\\\')\\n        models[h]=joblib.load(root/name)  # trusted project package only\\n        if models[h][\\\'horizon\\\']!=h or sorted(models[h][\\\'assets\\\'])!=manifest[\\\'assets\\\']:raise ValueError(\\\'Model horizon/assets mismatch\\\')\\n    return models,manifest\\n\',\'conditional_inference.py\',\'exec\'),I.__dict__)\nL=types.ModuleType(\'price_lineage\')\nexec(compile(\'"""Accept daily bridge IDs only after replaying checksum-certified KRX evidence.\\n\\nNever changes price values, quality flags, or adjustment_version labels.\\n"""\\nimport hashlib\\nimport io\\nimport json\\nimport re\\nimport numpy as np\\nimport pandas as pd\\n\\ndef price_fingerprint(raw):\\n    frame=raw.copy()\\n    frame[\\\'trading_date\\\']=pd.to_datetime(frame.trading_date).dt.normalize()\\n    columns=[\\\'trading_date\\\',\\\'asset_id\\\',\\\'close\\\',\\\'volume\\\',\\\'adjustment_version\\\',\\\'source_ref\\\',\\n             \\\'price_basis\\\',\\\'quality_status\\\',\\\'is_mock\\\']\\n    return hashlib.sha256(pd.util.hash_pandas_object(\\n        frame.sort_values([\\\'trading_date\\\',\\\'asset_id\\\'])[columns],index=False).values.tobytes()).hexdigest()\\n\\ndef verify_lineage(raw, reference, documents, source_digests, reference_digest):\\n    raw=raw.copy();raw[\\\'trading_date\\\']=pd.to_datetime(raw.trading_date).dt.normalize()\\n    reference=reference.copy();reference[\\\'trading_date\\\']=pd.to_datetime(reference.trading_date).dt.normalize()\\n    if raw.duplicated([\\\'trading_date\\\',\\\'asset_id\\\']).any() or reference.duplicated([\\\'trading_date\\\',\\\'asset_id\\\']).any():\\n        raise ValueError(\\\'Duplicate lineage keys\\\')\\n    versions=sorted(raw.adjustment_version.dropna().unique())\\n    bases=[v for v in versions if re.fullmatch(r\\\'krx-[0-9]{8}-verified-bridge-through-[0-9]{8}\\\',v)]\\n    if len(bases)!=1:raise ValueError(\\\'A unique certified baseline is required\\\')\\n    base=bases[0];prefix=base.split(\\\'-verified-\\\')[0]+\\\'-bridge-\\\'\\n    cutoff=pd.to_datetime(base.rsplit(\\\'-\\\',1)[1],format=\\\'%Y%m%d\\\')\\n    baseline=raw.loc[raw.adjustment_version.eq(base)]\\n    if baseline.empty or baseline.trading_date.max()!=cutoff:raise ValueError(\\\'Certified baseline boundary mismatch\\\')\\n    if not baseline.trading_date.le(cutoff).all():raise ValueError(\\\'Baseline dates exceed declared boundary\\\')\\n    joined=baseline.merge(reference,on=[\\\'trading_date\\\',\\\'asset_id\\\'],how=\\\'left\\\',suffixes=(\\\'\\\',\\\'_reference\\\'),validate=\\\'one_to_one\\\')\\n    if joined.close_reference.isna().any():raise ValueError(\\\'Baseline reference missing\\\')\\n    np.testing.assert_allclose(joined.close,joined.close_reference,rtol=0,atol=1e-8)\\n    np.testing.assert_allclose(joined.volume,joined.volume_reference,rtol=0,atol=0)\\n    extension=raw.loc[~raw.adjustment_version.eq(base)]\\n    if extension.empty:raise ValueError(\\\'No bridge extension to certify\\\')\\n    for version,group in extension.groupby(\\\'adjustment_version\\\'):\\n        if not re.fullmatch(re.escape(prefix)+r\\\'[0-9]{8}\\\',version):raise ValueError(\\\'Unrecognized bridge lineage\\\')\\n        day=pd.to_datetime(version[-8:],format=\\\'%Y%m%d\\\')\\n        if not group.trading_date.eq(day).all() or not day>cutoff:raise ValueError(\\\'Bridge version date mismatch\\\')\\n    assets=set(raw.asset_id)\\n    for day,group in extension.groupby(\\\'trading_date\\\'):\\n        if len(group)!=len(assets) or set(group.asset_id)!=assets:raise ValueError(\\\'Incomplete daily bridge\\\')\\n    if not extension.quality_status.eq(\\\'valid\\\').all() or extension.is_mock.any() or not raw.price_basis.eq(\\\'adjusted\\\').all():\\n        raise ValueError(\\\'Invalid bridge quality or price basis\\\')\\n    source_rows={}\\n    for path,doc in documents.items():\\n        if doc.get(\\\'is_mock\\\') is not False:raise ValueError(\\\'Uncertified/mock KRX response\\\')\\n        for row in doc.get(\\\'rows\\\',[]):\\n            aid=str(row.get(\\\'ISU_SRT_CD\\\') or row.get(\\\'ISU_CD\\\'))\\n            if aid not in assets:continue\\n            day=pd.to_datetime(str(row[\\\'BAS_DD\\\']).replace(\\\'/\\\',\\\'\\\'),format=\\\'%Y%m%d\\\')\\n            values=tuple(float(str(row[k]).replace(\\\',\\\',\\\'\\\')) for k in (\\\'TDD_CLSPRC\\\',\\\'ACC_TRDVOL\\\',\\\'CMPPREVDD_PRC\\\',\\\'FLUC_RT\\\'))\\n            key=(path,aid,day)\\n            if key in source_rows and values!=source_rows[key]:raise ValueError(\\\'Conflicting official rows\\\')\\n            source_rows[key]=values\\n    checked=0\\n    for aid,group in raw.groupby(\\\'asset_id\\\'):\\n        old=group.loc[group.trading_date.le(cutoff)].sort_values(\\\'trading_date\\\')\\n        if old.empty or old.iloc[-1].trading_date!=cutoff:raise ValueError(\\\'Asset lacks verified boundary\\\')\\n        previous=float(old.iloc[-1].close)\\n        for row in group.loc[group.trading_date.gt(cutoff)].sort_values(\\\'trading_date\\\').itertuples():\\n            key=(row.source_ref,aid,row.trading_date)\\n            if key not in source_rows:raise ValueError(\\\'Official daily receipt missing: \\\'+aid+\\\' \\\'+str(row.trading_date.date()))\\n            close,volume,change,pct=source_rows[key]\\n            if (close<=0 or volume<0 or abs(close-float(row.close))>1e-8 or volume!=float(row.volume)\\n                or abs((close-change)-previous)>.51 or abs((close/previous-1)*100-pct)>.011):\\n                raise ValueError(\\\'Corporate action or inconsistent bridge: \\\'+aid+\\\' \\\'+str(row.trading_date.date()))\\n            previous=close;checked+=1\\n    return dict(status=\\\'PASS\\\',baseline=base,baseline_rows=len(baseline),bridge_rows=checked,\\n                reference_sha256=reference_digest,receipt_sha256=dict(sorted(source_digests.items())),\\n                adjustment_versions=versions,latest_date=str(raw.trading_date.max().date()),\\n                gold_price_sha256=price_fingerprint(raw),\\n                labels_preserved=True,price_values_preserved=True)\\n\\ndef read_fabric_evidence(spark,raw,table):\\n    """Read existing OneLake receipts and their manifests; no cloud writes."""\\n    location=spark.sql(\\\'DESCRIBE DETAIL \\\'+table).select(\\\'location\\\').first()[0]\\n    if \\\'/Tables/\\\' not in location:raise ValueError(\\\'Unrecognized Lakehouse table location\\\')\\n    root=location.split(\\\'/Tables/\\\')[0]+\\\'/Files\\\'\\n    def blobs(paths):\\n        if not paths:return {}\\n        return {r.path:bytes(r.content) for r in spark.read.format(\\\'binaryFile\\\').load(list(paths)).select(\\\'path\\\',\\\'content\\\').collect()}\\n    reference_root=root+\\\'/reference/ml_input_validation/20261109_v1\\\'\\n    manifest_path=reference_root+\\\'/manifest.json\\\'\\n    content=blobs([manifest_path])[manifest_path];manifest=json.loads(content)\\n    expected=manifest[\\\'files\\\'][\\\'verified_prices.parquet\\\']\\n    reference_path=reference_root+\\\'/verified_prices.parquet\\\'\\n    reference_bytes=blobs([reference_path])[reference_path]\\n    if hashlib.sha256(reference_bytes).hexdigest()!=expected:raise ValueError(\\\'Certified reference SHA mismatch\\\')\\n    reference=pd.read_parquet(io.BytesIO(reference_bytes))\\n    extended=raw.loc[raw.adjustment_version.str.match(r\\\'^krx-[0-9]{8}-bridge-[0-9]{8}$\\\',na=False)]\\n    refs=set(extended.source_ref)\\n    bronze=root+\\\'/bronze/\\\'\\n    manifests={}\\n    for path in refs:\\n        if not isinstance(path,str) or not path.startswith(bronze+\\\'krx/api/\\\') or \\\'/stocks/\\\' not in path:\\n            raise ValueError(\\\'Bridge source outside certified KRX tree\\\')\\n        relative=path[len(bronze):]\\n        if \\\'..\\\' in relative.split(\\\'/\\\'):raise ValueError(\\\'Unsafe source path\\\')\\n        batch=relative.split(\\\'/\\\')[2]\\n        manifests[path]=bronze+\\\'_runs/\\\'+batch+\\\'/SUCCESS.json\\\'\\n    all_blobs=blobs(refs|set(manifests.values()))\\n    docs={};digests={}\\n    for path,manifest_path in manifests.items():\\n        receipt=json.loads(all_blobs[manifest_path])\\n        if receipt.get(\\\'status\\\') not in (\\\'complete\\\',\\\'no_market_observations\\\'):raise ValueError(\\\'Incomplete source batch\\\')\\n        relative=path[len(bronze):]\\n        matches=[r[\\\'sha256\\\'] for r in receipt.get(\\\'files\\\',[]) if r[\\\'file\\\'].replace(\\\'\\\\\\\\\\\',\\\'/\\\')==relative]\\n        digest=hashlib.sha256(all_blobs[path]).hexdigest()\\n        if len(set(matches))!=1 or matches[0]!=digest:raise ValueError(\\\'KRX receipt checksum mismatch\\\')\\n        docs[path]=json.loads(all_blobs[path]);digests[path]=digest\\n    return verify_lineage(raw,reference,docs,digests,expected)\\n\',\'price_lineage.py\',\'exec\'),L.__dict__)\nT=types.ModuleType(\'conditional_training\')\nexec(compile(\'"""Train-only, chronological two-stage fitting for the pooled conditional model.\\n\\n250 observations per asset remain required over the whole Train set. Requiring\\nall 250 in the first stage is inappropriate for this shared 41-feature model:\\nnew listings can exhaust the second stage. Fixed feasibility rules below are\\nchecked without looking at Calibration/Test targets or their performance.\\n"""\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.pipeline import make_pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.linear_model import HuberRegressor\\nfrom catboost import CatBoostRegressor\\n\\nSTRATEGY=\\\'balanced_chronological_two_stage_v2\\\'\\nRULES=dict(min_total_rows_per_asset=250,min_huber_rows_per_asset=100,\\n           min_catboost_rows_per_asset=60,min_catboost_dates=120,\\n           min_rows_per_stage=1000,preferred_huber_fraction=.8)\\n\\ndef choose_split(train,assets):\\n    train=train.sort_values([\\\'trading_date\\\',\\\'asset_id\\\']).copy()\\n    if train[[\\\'trading_date\\\',\\\'target_date\\\',\\\'asset_id\\\']].isna().any().any():\\n        raise ValueError(\\\'Training keys/target dates are null\\\')\\n    if train.duplicated([\\\'trading_date\\\',\\\'asset_id\\\']).any():raise ValueError(\\\'Duplicate training keys\\\')\\n    if not train.target_date.gt(train.trading_date).all():raise ValueError(\\\'Invalid target horizon\\\')\\n    counts=train.groupby(\\\'asset_id\\\').size()\\n    if set(counts.index)!=set(assets) or counts.min()<RULES[\\\'min_total_rows_per_asset\\\']:\\n        raise ValueError(\\\'Whole Train must retain at least 250 rows for each of the 20 assets\\\')\\n    dates=pd.DatetimeIndex(sorted(train.trading_date.unique()))\\n    if len(dates)<RULES[\\\'min_catboost_dates\\\']:raise ValueError(\\\'Fewer than 120 Train dates\\\')\\n    original=dates[int(len(dates)*RULES[\\\'preferred_huber_fraction\\\'])]\\n    # First-stage labels must have matured strictly before the second stage.\\n    lower=max(g.target_date.sort_values().iloc[RULES[\\\'min_huber_rows_per_asset\\\']-1]\\n              for _,g in train.groupby(\\\'asset_id\\\'))\\n    upper=min(dates[-RULES[\\\'min_catboost_dates\\\']],\\n              min(g.trading_date.sort_values().iloc[-RULES[\\\'min_catboost_rows_per_asset\\\']]\\n                  for _,g in train.groupby(\\\'asset_id\\\')))\\n    eligible=dates[(dates>lower)&(dates<=upper)]\\n    if eligible.empty:\\n        raise ValueError(\\\'No feasible Train-only stage boundary: require 100 matured Huber rows per asset, \\\'\\n                         \\\'60 CatBoost rows per asset and 120 distinct CatBoost dates. \\\'\\n                         f\\\'Huber boundary must be after {lower}; CatBoost boundary no later than {upper}. \\\'\\n                         \\\'Do not borrow Calibration/Test rows or lower these rules silently.\\\')\\n    cutoff=min(eligible,key=lambda d:abs(d-original))\\n    first=train.loc[train.target_date<cutoff].copy()\\n    second=train.loc[train.trading_date>=cutoff].copy()\\n    first_counts=first.groupby(\\\'asset_id\\\').size().reindex(assets,fill_value=0)\\n    second_counts=second.groupby(\\\'asset_id\\\').size().reindex(assets,fill_value=0)\\n    if (first_counts.min()<RULES[\\\'min_huber_rows_per_asset\\\']\\n        or second_counts.min()<RULES[\\\'min_catboost_rows_per_asset\\\']\\n        or second.trading_date.nunique()<RULES[\\\'min_catboost_dates\\\']\\n        or min(len(first),len(second))<RULES[\\\'min_rows_per_stage\\\']):\\n        raise ValueError(\\\'Internal stage feasibility assertion failed\\\')\\n    if not first.target_date.max()<second.trading_date.min():raise ValueError(\\\'Training stages overlap\\\')\\n    summary=dict(training_strategy=STRATEGY,original_cutoff=str(original.date()),\\n        inner_cutoff=str(cutoff.date()),huber_rows=len(first),catboost_rows=len(second),\\n        huber_label_end=str(first.target_date.max().date()),\\n        catboost_start=str(second.trading_date.min().date()),\\n        huber_assets=int(first.asset_id.nunique()),catboost_assets=int(second.asset_id.nunique()),\\n        huber_min_asset_rows=int(first_counts.min()),catboost_min_asset_rows=int(second_counts.min()),\\n        catboost_dates=int(second.trading_date.nunique()),purged_between_stages=len(train)-len(first)-len(second))\\n    return first,second,summary\\n\\ndef fit(ctx,inference,asym_halfwidths):\\n    first,second,summary=choose_split(ctx.train,ctx.assets)\\n    scale=lambda frame:np.maximum(frame.asset_scale.to_numpy(float),.05)\\n    huber=make_pipeline(StandardScaler(),HuberRegressor(alpha=20.,epsilon=1.35,max_iter=2000))\\n    huber.fit(inference.design(first),first[ctx.ycol].to_numpy()/scale(first))\\n    residual=second[ctx.ycol].to_numpy()/scale(second)-huber.predict(inference.design(second))\\n    cat=CatBoostRegressor(loss_function=\\\'RMSEWithUncertainty\\\',iterations=300,depth=4,\\n        learning_rate=.03,l2_leaf_reg=20,random_seed=20261105,thread_count=4,\\n        verbose=False,allow_writing_files=False)\\n    cat.fit(inference.design(second),residual)\\n    x=inference.design(ctx.cal);raw=cat.predict(x,prediction_type=\\\'RawFormulaVal\\\')\\n    mu=(huber.predict(x)+raw[:,0])*scale(ctx.cal)\\n    sigma=np.exp(raw[:,1])*scale(ctx.cal)\\n    if not np.isfinite(sigma).all() or (sigma<=0).any():raise ValueError(\\\'Invalid calibration scale\\\')\\n    errors=(ctx.cal[ctx.ycol].to_numpy()-mu)/sigma\\n    bands={level:asym_halfwidths(errors,level) for level in (.8,.9,.95)}\\n    if not np.isfinite(list(bands.values())).all():raise ValueError(\\\'Insufficient calibration residuals\\\')\\n    return dict(huber=huber,catboost=cat,bands=bands,horizon=ctx.h,assets=ctx.assets,\\n                kind=\\\'Huber+CatBoost\\\',stage_rules=RULES.copy(),**summary)\\n\',\'conditional_training.py\',\'exec\'),T.__dict__)\n"""Functions appended to the original, checksum-verified source bootstrap."""\nfrom uuid import uuid4\nimport shutil\n\nASSETS=sorted([\'000270\',\'000660\',\'005380\',\'005930\',\'005935\',\'006400\',\'009150\',\'012330\',\'012450\',\'028260\',\n               \'032830\',\'034020\',\'034730\',\'055550\',\'068270\',\'086790\',\'105560\',\'207940\',\'373220\',\'402340\'])\nHORIZONS=(1,5,20)\nMODEL=\'huber_catboost_hybrid_conditional\'\nSTATE_FIELDS=I.STATE_FIELDS\n\ndef sha256(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\ndef encoded(data):return (json.dumps(data,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)+\'\\n\').encode(\'utf-8\')\ndef write_json(path,data,immutable=False):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);content=encoded(data)\n    if path.exists() and immutable:\n        if path.read_bytes()!=content:raise ValueError(\'Immutable file collision\')\n        return\n    tmp=path.with_name(path.name+\'.\'+uuid4().hex+\'.tmp\');tmp.write_bytes(content);os.replace(tmp,path)\n\ndef load_checked_gold(raw,dataset_version,price_lineage=None):\n    raw=raw.loc[raw.dataset_version.eq(dataset_version)].copy()\n    if raw.empty:raise ValueError(\'Gold dataset version missing\')\n    # Keep original labels and every validation other than the certified\n    # single-version assumption. Never disable Gold validation as a whole.\n    gold,report=load_gold(raw,schema_path=SCHEMA,return_report=True,raise_on_error=False)\n    mixed=any(i.check==\'adjustment_version_mixed\' for i in report.errors)\n    if mixed:\n        if not price_lineage or price_lineage.get(\'status\')!=\'PASS\':\n            raise ValueError(\'Mixed adjustment versions require a freshly verified price lineage\')\n        if (price_lineage.get(\'labels_preserved\') is not True\n            or price_lineage.get(\'price_values_preserved\') is not True\n            or price_lineage.get(\'gold_price_sha256\')!=L.price_fingerprint(raw)\n            or price_lineage.get(\'adjustment_versions\')!=sorted(raw.adjustment_version.dropna().unique())):\n            raise ValueError(\'Price lineage does not match the current Gold rows\')\n    remaining=[i for i in report.errors if not (mixed and i.check==\'adjustment_version_mixed\')]\n    if remaining:\n        from load_gold import GoldValidationError,GoldReport\n        raise GoldValidationError(GoldReport(report.mode,remaining,report.summary,report.exclusions))\n    gold.attrs[\'price_lineage\']=price_lineage if mixed else None\n    return gold\n\ndef prepare(raw,dataset_version,price_lineage=None):\n    gold=load_checked_gold(raw,dataset_version,price_lineage)\n    clean,features,market=C.load_inputs(gold)\n    if set(clean.asset_id)!=set(ASSETS):raise ValueError(\'20-asset contract mismatch\')\n    contexts={}\n    from prep import rolling_beta_frames\n    from models import add_beta\n    beta_frames,kospi_prices,_=rolling_beta_frames(clean)\n    for h in HORIZONS:\n        ctx=C.Context(clean,features,market,h,calibration_start=\'2024-01-01\',evaluation_start=\'2025-01-01\')\n        if ctx.assets!=ASSETS:raise ValueError(f\'{h}d eligible assets mismatch\')\n        for split in (\'all\',\'train\',\'cal\',\'test\'):\n            frame=getattr(ctx,split).copy()\n            for name,values in beta_frames.items():\n                stacked=values.stack();stacked.index.names=[\'trading_date\',\'asset_id\']\n                frame[name]=stacked.reindex(pd.MultiIndex.from_frame(frame[[\'trading_date\',\'asset_id\']])).to_numpy()\n            # Realized future KOSPI is used only for historical conditional fitting/evaluation.\n            frame[\'m\']=100*np.log(kospi_prices.reindex(frame.target_date).to_numpy()/kospi_prices.reindex(frame.trading_date).to_numpy())\n            frame[\'horizon\']=h;frame[\'idio_missing\']=frame.idio_vol60.isna()\n            setattr(ctx,split,frame)\n        fill=ctx.train.idio_vol60.median()\n        if not np.isfinite(fill):raise ValueError(\'Training-only idiosyncratic scale unavailable\')\n        for split in (\'all\',\'train\',\'cal\',\'test\'):\n            frame=getattr(ctx,split);frame[\'idio_vol60\']=frame.idio_vol60.fillna(fill)\n            setattr(ctx,split,add_beta(frame))\n        ctx.idio_fill=float(fill)\n        for split in (\'train\',\'cal\',\'test\'):\n            frame=getattr(ctx,split)\n            if frame.m.isna().any() or set(frame.asset_id)!=set(ASSETS):raise ValueError(\'Incomplete conditional split\')\n        contexts[h]=ctx\n    canonical=gold.sort_values([\'trading_date\',\'asset_id\']).reset_index(drop=True)\n    digest=hashlib.sha256(pd.util.hash_pandas_object(canonical,index=False).values.tobytes()).hexdigest()\n    return gold,contexts,digest\n\ndef training_metrics(pred):\n    rows=[];y=pred.y_actual.to_numpy()\n    for name,value in [(MODEL,pred.center.to_numpy()),(\'huber_only\',pred.center_huber.to_numpy()),(\'zero_return\',np.zeros(len(pred)))]:\n        row=dict(model=name,rows=len(pred),**C.point_metrics(y,value))\n        if name==MODEL:\n            for tag in (80,90,95):\n                values=C.interval_metrics(y,pred[f\'lo_{tag}\'].to_numpy(),pred[f\'hi_{tag}\'].to_numpy(),tag/100)\n                row[f\'coverage{tag}\']=values[\'coverage\'];row[f\'width{tag}\']=values[\'mean_width\']\n        rows.append(row)\n    # A zero-return benchmark has undefined nonzero directional accuracy.\n    # Keep strict JSON and represent that statistic as null, never NaN.\n    return [{k:(None if isinstance(v,(float,np.floating)) and not np.isfinite(v) else v)\n             for k,v in row.items()} for row in rows]\n\ndef train_run(gold,contexts,gold_sha,run_dir,dataset_version):\n    core=importlib.import_module(\'huber_catboost_hybrid_core\')\n    root=Path(run_dir);root.mkdir(parents=True,exist_ok=False)\n    rows=[];stage_rows=[]\n    for h,ctx in contexts.items():\n        from conformal import asym_halfwidths\n        model=T.fit(ctx,I,asym_halfwidths)\n        p=core.predict(model,ctx.test)\n        # Independently verify the inference-only 41-column implementation.\n        np.testing.assert_allclose(p,I.predict(model,ctx.test),rtol=1e-12,atol=1e-12)\n        joblib.dump(model,root/f\'model_{h}d.joblib\',compress=3)\n        np.testing.assert_allclose(p,I.predict(joblib.load(root/f\'model_{h}d.joblib\'),ctx.test),atol=1e-12)\n        z=ctx.test[[\'trading_date\',\'target_date\',\'asset_id\',ctx.ycol]].rename(columns={ctx.ycol:\'y_actual\'}).join(p).assign(horizon=h)\n        z.to_parquet(root/f\'predictions_{h}d.parquet\',index=False)\n        rows += [dict(horizon=h,**m) for m in training_metrics(z)]\n        stage_rows.append(dict(horizon=h,huber_rows=model[\'huber_rows\'],catboost_rows=model[\'catboost_rows\'],\n            huber_target_end=model[\'huber_label_end\'],catboost_start=model[\'catboost_start\'],\n            training_strategy=model[\'training_strategy\'],original_cutoff=model[\'original_cutoff\'],\n            adjusted_cutoff=model[\'inner_cutoff\'],huber_min_asset_rows=model[\'huber_min_asset_rows\'],\n            catboost_min_asset_rows=model[\'catboost_min_asset_rows\'],catboost_dates=model[\'catboost_dates\']))\n        print(\'Conditional model trained:\',h,\'days;\',len(z),\'test rows\',flush=True)\n    pd.DataFrame(rows).to_csv(root/\'metrics.csv\',index=False)\n    pd.DataFrame(stage_rows).to_csv(root/\'training_stages.csv\',index=False)\n    versions={k:importlib.metadata.version(k) for k in (\'numpy\',\'pandas\',\'scipy\',\'scikit-learn\',\'catboost\',\'joblib\',\'pyarrow\')}\n    manifest=dict(model=MODEL,display_name=\'HAB-input Huber + CatBoost conditional hybrid\',\n        dataset_version=dataset_version,source_sha256=SOURCE_SHA256,gold_sha256=gold_sha,\n        price_lineage=gold.attrs.get(\'price_lineage\'),\n        assets=ASSETS,horizons=list(HORIZONS),design_feature_count=41,\n        training_strategy=T.STRATEGY,stage_rules=T.RULES.copy(),\n        research_only=True,production_approval=\'NOT_APPROVED\',historical_availability_verified=False,\n        future_market_or_macro_input=True,future_input_usage=\'realized_for_historical_evaluation; assumptions_for_service\',\n        versions=versions,calibration_start=\'2024-01-01\',evaluation_start=\'2025-01-01\',\n        idio_fill={str(h):c.idio_fill for h,c in contexts.items()},\n        splits={str(h):c.summary() for h,c in contexts.items()},\n        files={p.name:sha256(p) for p in root.iterdir() if p.is_file()})\n    write_json(root/\'manifest.json\',manifest,True)\n    return manifest\n\ndef verify(root):\n    root=Path(root);m=json.loads((root/\'manifest.json\').read_text(encoding=\'utf-8\'))\n    if m[\'model\']!=MODEL or m[\'future_market_or_macro_input\'] is not True or m[\'assets\']!=ASSETS or m[\'horizons\']!=list(HORIZONS):raise ValueError(\'Wrong conditional model contract\')\n    if m[\'production_approval\']!=\'NOT_APPROVED\' or m[\'research_only\'] is not True:raise ValueError(\'Wrong approval status\')\n    for name,expected in m[\'files\'].items():\n        if Path(name).name!=name or sha256(root/name)!=expected:raise ValueError(\'Artifact changed: \'+name)\n    return m\n\ndef validate_run(root,contexts,gold_sha):\n    root=Path(root);m=verify(root)\n    if gold_sha!=m[\'gold_sha256\']:raise ValueError(\'Gold changed since training\')\n    rows=[]\n    for h,ctx in contexts.items():\n        model=joblib.load(root/f\'model_{h}d.joblib\')\n        if not pd.Timestamp(model[\'huber_label_end\'])<pd.Timestamp(model[\'catboost_start\'])<pd.Timestamp(\'2024-01-01\'):raise ValueError(\'Training time boundary failure\')\n        if m.get(\'training_strategy\')==T.STRATEGY:\n            _,_,expected_stage=T.choose_split(ctx.train,ctx.assets)\n            if model.get(\'stage_rules\')!=T.RULES or any(model.get(k)!=v for k,v in expected_stage.items()):\n                raise ValueError(\'Recorded balanced training stages do not match Train-only reconstruction\')\n        if not ctx.train.target_date.lt(\'2024-01-01\').all() or not ctx.cal.target_date.lt(\'2025-01-01\').all() or not ctx.test.trading_date.ge(\'2025-01-01\').all():raise ValueError(\'Split boundary failure\')\n        saved=pd.read_parquet(root/f\'predictions_{h}d.parquet\')\n        for key in (\'trading_date\',\'target_date\',\'asset_id\'):np.testing.assert_array_equal(saved[key],ctx.test[key])\n        np.testing.assert_allclose(saved.y_actual,ctx.test[ctx.ycol])\n        p=I.predict(model,ctx.test);np.testing.assert_allclose(saved[list(p)],p,rtol=1e-10,atol=1e-10)\n        if len(saved)!=m[\'splits\'][str(h)][\'test_rows\'] or set(saved.asset_id)!=set(ASSETS):raise ValueError(\'Evaluation cardinality failure\')\n        rows += [dict(horizon=h,**v) for v in training_metrics(saved)]\n    return dict(status=\'PASS\',validation_type=\'conditional research technical validation\',\n                manifest_sha256=sha256(root/\'manifest.json\'),metrics=rows,\n                future_market_or_macro_input=True,production_approval=\'NOT_APPROVED\')\n\ndef make_state_snapshot(gold,contexts,descriptor,manifest):\n    asof=gold.trading_date.max();states={}\n    if pd.Timestamp(asof).date()>datetime.now(timezone.utc).date():raise ValueError(\'Future Gold date\')\n    prices=gold.loc[gold.trading_date.eq(asof)].set_index(\'asset_id\')\n    if set(prices.index)!=set(ASSETS) or not prices.quality_status.eq(\'valid\').all() or prices.is_mock.any():raise ValueError(\'Invalid latest prices\')\n    for h,ctx in contexts.items():\n        frame=ctx.all.loc[ctx.all.trading_date.eq(asof)].copy()\n        if len(frame)!=20 or set(frame.asset_id)!=set(ASSETS):raise ValueError(\'Missing latest conditional state\')\n        # Apply the persisted training-only fill, not a median from current/live rows.\n        frame[\'idio_f\']=frame.idio_vol60.where(~frame.idio_missing,manifest[\'idio_fill\'][str(h)])\n        if not np.isfinite(frame[STATE_FIELDS].to_numpy(float)).all():raise ValueError(\'Latest state has missing features\')\n        records=[]\n        for _,row in frame.sort_values(\'asset_id\').iterrows():\n            records.append(dict(asset_id=row.asset_id,horizon=h,base_close=float(prices.loc[row.asset_id,\'close\']),\n                                **{c:float(row[c]) for c in STATE_FIELDS}))\n        states[str(h)]=records\n    return dict(snapshot_version=\'conditional-v4\',model=MODEL,model_package_id=descriptor[\'package_id\'],\n                model_zip_sha256=descriptor[\'sha256\'],dataset_version=manifest[\'dataset_version\'],\n                as_of_trading_date=str(pd.Timestamp(asof).date()),price_basis=\'adjusted\',\n                research_only=True,production_approval=\'NOT_APPROVED\',real_time=False,\n                historical_availability_verified=False,state_by_horizon=states,\n                note=\'State only. Future market moves must be supplied as scenario assumptions.\')\n\ndef package_run(root,destination,inference_source):\n    root=Path(root);m=verify(root)\n    report=json.loads((root/\'validation.json\').read_text(encoding=\'utf-8\'))\n    if report[\'status\']!=\'PASS\' or report[\'manifest_sha256\']!=sha256(root/\'manifest.json\'):raise ValueError(\'NB_07 validation missing/stale\')\n    files={name:(root/name).read_bytes() for name in [\'manifest.json\',\'validation.json\']+[f\'model_{h}d.joblib\' for h in HORIZONS]}\n    files[\'conditional_inference.py\']=inference_source.encode()\n    files[\'requirements.txt\']=\'\\n\'.join(k+\'==\'+v for k,v in m[\'versions\'].items() if k not in (\'scipy\',\'pyarrow\')).encode()\n    digest=hashlib.sha256(b\'\'.join(k.encode()+hashlib.sha256(v).digest() for k,v in sorted(files.items()))).hexdigest()\n    package_id=\'hab-conditional-v4-\'+digest[:16]\n    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)\n    path=destination/(package_id+\'.zip\');temporary=path.with_name(path.name+\'.\'+uuid4().hex+\'.tmp\')\n    files[\'package.json\']=encoded(dict(package_id=package_id,files={k:hashlib.sha256(v).hexdigest() for k,v in files.items()}))\n    with zipfile.ZipFile(temporary,\'w\') as z:\n        for name,content in sorted(files.items()):\n            info=zipfile.ZipInfo(name,(2026,1,1,0,0,0));info.compress_type=zipfile.ZIP_DEFLATED;z.writestr(info,content)\n    if path.exists() and sha256(path)!=sha256(temporary):raise ValueError(\'Package collision\')\n    os.replace(temporary,path)\n    descriptor=dict(package_id=package_id,path=str(path),sha256=sha256(path),manifest_sha256=sha256(root/\'manifest.json\'))\n    write_json(destination/(package_id+\'.json\'),descriptor,True)\n    return descriptor\n\ndef load_package(descriptor):\n    if sha256(descriptor[\'path\'])!=descriptor[\'sha256\']:raise ValueError(\'Package SHA mismatch\')\n    work=Path(tempfile.mkdtemp(prefix=\'conditional_models_\'))\n    with zipfile.ZipFile(descriptor[\'path\']) as z:\n        names=z.namelist();contract=json.loads(z.read(\'package.json\'))\n        if contract[\'package_id\']!=descriptor[\'package_id\'] or set(names)!=set(contract[\'files\'])|{\'package.json\'} or len(names)!=len(set(names)):raise ValueError(\'Package contract mismatch\')\n        for name,digest in contract[\'files\'].items():\n            if Path(name).name!=name or \'\\\\\' in name:raise ValueError(\'Unsafe package path\')\n            content=z.read(name)\n            if hashlib.sha256(content).hexdigest()!=digest:raise ValueError(\'Package member SHA mismatch\')\n            (work/name).write_bytes(content)\n    return I.load_models(work,descriptor[\'manifest_sha256\'])\n'
RUNTIME_SHA256='f0cfe9b0ea169491d068fd2c5de8a645c184d84581eed3eef43cbd3a605c3f7a'
if hashlib.sha256(SOURCE.encode()).hexdigest()!=RUNTIME_SHA256:raise ValueError('Runtime source changed')
R=types.ModuleType('conditional_runtime')
sys.modules[R.__name__]=R
exec(compile(SOURCE,'conditional_runtime.py','exec'),R.__dict__)
REF_LOADER=types.ModuleType('conditional_reference_loader')
exec(compile('"""Load a project\'s trusted, recorded NB_05 runtime; never waive its digest.\n\nThe source file must be saved by the exact NB_05 that trained the referenced\nmodels. A changed manifest or source stops execution before dynamic loading.\n"""\nfrom pathlib import Path\nimport hashlib,json,re,sys,types,ast\n\ndef resolve(reference,current_runtime,current_sha):\n    root=Path(reference[\'run_dir\'])\n    manifest=root/\'manifest.json\'\n    if hashlib.sha256(manifest.read_bytes()).hexdigest()!=reference[\'manifest_sha256\']:\n        raise ValueError(\'Run manifest changed\')\n    expected=reference[\'runtime_sha256\']\n    if not re.fullmatch(\'[0-9a-f]{64}\',expected):raise ValueError(\'Invalid run runtime digest\')\n    if expected==current_sha:return current_runtime,expected,None\n    path=root/\'runtime_source.py\'\n    if not path.is_file():\n        raise ValueError(\'Runtime versions differ. Save runtime_source.py from the NB_05 that completed training. \'\n                         f\'NB_05={expected}; current={current_sha}. Do not retrain or disable the digest check.\')\n    source=path.read_text(encoding=\'utf-8\')\n    if hashlib.sha256(source.encode(\'utf-8\')).hexdigest()!=expected:\n        raise ValueError(\'Recorded NB_05 runtime source SHA mismatch\')\n    module=types.ModuleType(\'conditional_runtime_recorded_\'+expected[:16])\n    sys.modules[module.__name__]=module\n    exec(compile(source,str(path),\'exec\'),module.__dict__)\n    module.verify(root)\n    return module,expected,source\n\ndef inference_source(runtime_source):\n    candidates=[]\n    for node in ast.walk(ast.parse(runtime_source)):\n        if (isinstance(node,ast.Call) and isinstance(node.func,ast.Name) and node.func.id==\'compile\'\n            and len(node.args)>=2 and isinstance(node.args[1],ast.Constant)\n            and node.args[1].value==\'conditional_inference.py\'):\n            value=ast.literal_eval(node.args[0])\n            if not isinstance(value,str):raise ValueError(\'Invalid recorded inference source\')\n            candidates.append(value)\n    if len(candidates)!=1:raise ValueError(\'Expected exactly one recorded inference implementation\')\n    return candidates[0]\n','conditional_reference_loader.py','exec'),REF_LOADER.__dict__)
ROOT=Path('/lakehouse/default/Files/stresslab/conditional_v4')
ROOT.mkdir(parents=True,exist_ok=True)
def pipeline():return bool(notebookutils.runtime.context['isForPipeline'])
def finish(value):
    if pipeline():notebookutils.notebook.exit(value)
def read_gold(table,version):
    spark.conf.set('spark.sql.session.timeZone','UTC')
    frame=spark.read.table(table)
    if any(f.dataType.simpleString()=='timestamp_ntz' for f in frame.schema.fields):
        raise ValueError('Gold timestamp_ntz 원천 시간대를 먼저 확인하세요')
    raw=frame.toPandas()
    for field in frame.schema.fields:
        if field.dataType.simpleString()=='timestamp':raw[field.name]=R.pd.to_datetime(raw[field.name],utc=True,errors='raise')
    selected=raw.loc[raw.dataset_version.eq(version)].copy()
    lineage_proof=None
    if selected.groupby('asset_id').adjustment_version.nunique().gt(1).any():
        lineage_proof=R.L.read_fabric_evidence(spark,selected,table)
        print('Certified price lineage:',lineage_proof['status'],'bridge rows:',lineage_proof['bridge_rows'])
    return R.prepare(selected,version,price_lineage=lineage_proof)

In [ ]:
# 학습 당시 Gold를 재현하고 Train의 실제 관측만 별도 파일로 내보냅니다.
import gzip
import zipfile
from pyspark.sql import functions as F

package_path = Path(PACKAGE_PATH)
if R.sha256(package_path) != EXPECTED_ZIP_SHA256:
    raise ValueError("Package ZIP SHA mismatch")
with zipfile.ZipFile(package_path) as archive:
    manifest_bytes = archive.read("manifest.json")
    package_contract = json.loads(archive.read("package.json"))
if hashlib.sha256(manifest_bytes).hexdigest() != EXPECTED_MANIFEST_SHA256:
    raise ValueError("Manifest SHA mismatch")
manifest = json.loads(manifest_bytes)
if manifest["source_sha256"] != R.SOURCE_SHA256:
    raise ValueError("Original preprocessing source differs from the model")
if manifest["assets"] != R.ASSETS or manifest["horizons"] != [1,5,20]:
    raise ValueError("Model asset/horizon contract mismatch")

# Read the original training snapshot endpoint from its manifest, not today's date.
source_end = manifest["price_lineage"]["latest_date"]
spark.conf.set("spark.sql.session.timeZone", "UTC")
frame = spark.read.table(GOLD_TABLE)
if any(f.dataType.simpleString() == "timestamp_ntz" for f in frame.schema.fields):
    raise ValueError("Gold timestamp_ntz requires explicit source timezone validation")
frame = frame.filter((F.col("dataset_version") == manifest["dataset_version"]) &
                     (F.col("trading_date") <= F.to_date(F.lit(source_end))))
raw = frame.toPandas()
for field in frame.schema.fields:
    if field.dataType.simpleString() == "timestamp":
        raw[field.name] = R.pd.to_datetime(raw[field.name], utc=True, errors="raise")
lineage = None
if raw.groupby("asset_id").adjustment_version.nunique().gt(1).any():
    lineage = R.L.read_fabric_evidence(spark, raw, GOLD_TABLE)
gold, contexts, gold_sha = R.prepare(raw, manifest["dataset_version"], price_lineage=lineage)
if gold_sha != manifest["gold_sha256"]:
    raise ValueError("Original training Gold cannot be reproduced; do not export mixed historical data")

shocks = ["shock_usdkrw_pct","shock_us10y_bp","shock_wti_pct","shock_vix_pt"]
states = ["kospi_vol20","kospi_trend20","vix_level","us10y_level","usdkrw_trend20","wti_level"]
columns = ["asset_id","horizon_days","start","end","actual_return_pct"] + shocks + states
rows, counts, summary = [], {}, []
for h, ctx in contexts.items():
    train = ctx.train.sort_values(["trading_date","asset_id"]).copy()
    expected = manifest["splits"][str(h)]
    period = [str(train.trading_date.min().date()), str(train.trading_date.max().date())]
    if len(train) != expected["train_rows"] or period != expected["train_period"]:
        raise ValueError(f"{h}d training sample contract mismatch")
    if not train.target_date.lt(manifest["calibration_start"]).all():
        raise ValueError("Calibration/Test observations must not enter the history index")
    train["horizon_days"] = h
    train["start"] = train.trading_date.dt.strftime("%Y-%m-%d")
    train["end"] = train.target_date.dt.strftime("%Y-%m-%d")
    train["actual_return_pct"] = 100 * R.np.expm1(train[ctx.ycol] / 100)
    if not R.np.isfinite(train[["actual_return_pct"] + shocks + states].to_numpy(float)).all():
        raise ValueError("Nonfinite Train history inputs")
    counts[str(h)] = {str(k):int(v) for k,v in train.groupby("asset_id").size().items()}
    for values in train[columns].itertuples(index=False, name=None):
        rows.append([str(values[0]),int(values[1]),str(values[2]),str(values[3])] +
                    [float(v) for v in values[4:]])
    summary.append(dict(horizon=h,rows=len(train),assets=train.asset_id.nunique(),
                        first_date=period[0],last_date=period[1]))

payload = dict(schema="conditional-history-v1",source_split="train",
    model_package_id=package_contract["package_id"],model_manifest_sha256=EXPECTED_MANIFEST_SHA256,
    dataset_version=manifest["dataset_version"],gold_sha256=gold_sha,
    calibration_start=manifest["calibration_start"],assets=manifest["assets"],
    columns=columns,train_rows=counts,rows=rows)
content = gzip.compress(R.encoded(payload), compresslevel=6, mtime=0)
digest = hashlib.sha256(content).hexdigest()
output_dir = ROOT / "history" / package_contract["package_id"]
output_dir.mkdir(parents=True,exist_ok=True)
output_path = output_dir / ("history_" + digest + ".json.gz")
if output_path.exists() and output_path.read_bytes() != content:
    raise ValueError("Immutable history artifact collision")
if not output_path.exists():
    output_path.write_bytes(content)
descriptor = dict(path=str(output_path),sha256=digest,rows=len(rows),
    model_package_id=payload["model_package_id"],model_manifest_sha256=EXPECTED_MANIFEST_SHA256)
display(R.pd.DataFrame(summary))
print("HISTORY_DESCRIPTOR:",json.dumps(descriptor,ensure_ascii=False))
print("TRAIN HISTORY READY — no training or Gold/Delta/PostgreSQL updates performed")
